In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configurazione stile grafici
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['font.size'] = 10

In [3]:
# ============================================================================
# SEZIONE 1: CARICAMENTO DATI
# ============================================================================

def parse_array_string(s):
    """Converte stringa array in numpy array"""
    if pd.isna(s) or s == '':
        return np.array([])
    s = s.strip('[]')
    return np.array([float(x) for x in s.split() if x])

def load_experiment_data(results_path='./results'):
    """Carica tutti i dati degli esperimenti"""
    results_path = Path(results_path)
    experiments = []
    
    # Itera su tutti i tipi di loss
    for loss_type in ['bce', 'contrastive', 'triplet']:
        loss_path = results_path / loss_type / f'{loss_type}_experiments'
        if not loss_path.exists():
            continue
            
        # Trova tutti i modelli
        for model_dir in loss_path.iterdir():
            if not model_dir.is_dir():
                continue
                
            # Estrai informazioni dal nome
            parts = model_dir.name.split('_')
            
            # Trova gli indici dei dataset
            to_idx = -1
            for i, part in enumerate(parts):
                if part == 'to':
                    to_idx = i
                    break
            
            if to_idx == -1:
                continue
                
            # Estrai model_name, train_dataset, test_dataset
            model_name = '_'.join(parts[:to_idx-1])  # tutto prima del loss type
            train_dataset = parts[to_idx-1]  # dataset prima di 'to'
            test_dataset = parts[to_idx+1]   # dataset dopo 'to'
            
            # Carica metrics
            metrics_file = model_dir / f"{model_dir.name}_final_metrics.csv"
            history_file = model_dir / f"{model_dir.name}_history.csv"
            
            if metrics_file.exists() and history_file.exists():
                metrics = pd.read_csv(metrics_file)
                history = pd.read_csv(history_file)
                
                # Parse array fields
                for col in ['fpr', 'tpr', 'thresholds', 'genuine_vals', 'impostor_vals']:
                    if col in metrics.columns:
                        metrics[col] = metrics[col].apply(parse_array_string)
                
                experiments.append({
                    'model': model_name,
                    'loss': loss_type,
                    'train_dataset': train_dataset,
                    'test_dataset': test_dataset,
                    'scenario': f"{train_dataset}_to_{test_dataset}",
                    'metrics': metrics.iloc[0],
                    'history': history,
                    'path': model_dir
                })
    
    return experiments

# ============================================================================
# SEZIONE 2: ANALISI COMPARATIVA METRICHE
# ============================================================================

def create_metrics_comparison(experiments):
    """Crea DataFrame comparativo delle metriche principali"""
    data = []
    for exp in experiments:
        m = exp['metrics']
        data.append({
            'Model': exp['model'],
            'Loss': exp['loss'],
            'Train': exp['train_dataset'],
            'Test': exp['test_dataset'],
            'Scenario': exp['scenario'],
            'AUC': m['auc'],
            'EER': m['eer'],
            'Accuracy': m['accuracy'],
            'F1': m['f1'],
            "D'": m['d_prime'],
            'Decidability': m['decidability'],
            'GAR@FAR=0.1%': m['gar_at_far_0.001'],
            'GAR@FAR=1%': m['gar_at_far_0.01'],
        })
    return pd.DataFrame(data)

def plot_metrics_comparison(df_metrics):
    """Plot comparativo delle metriche principali"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Confronto Metriche per Loss Function e Scenario', fontsize=16, fontweight='bold')
    
    metrics = ['AUC', 'EER', 'Accuracy', 'F1', "D'", 'GAR@FAR=0.1%']
    
    for idx, metric in enumerate(metrics):
        ax = axes[idx // 3, idx % 3]
        
        # Crea pivot per heatmap
        pivot = df_metrics.pivot_table(
            values=metric, 
            index='Scenario', 
            columns='Loss',
            aggfunc='mean'
        )
        
        # Plot heatmap
        sns.heatmap(pivot, annot=True, fmt='.4f', cmap='RdYlGn', 
                   ax=ax, cbar_kws={'label': metric})
        ax.set_title(f'{metric} per Scenario e Loss', fontweight='bold')
        ax.set_xlabel('Loss Function')
        ax.set_ylabel('Scenario')
        
        # Evidenzia il migliore
        if metric != 'EER':  # Per EER più basso è meglio
            best_val = pivot.max().max()
        else:
            best_val = pivot.min().min()
    
    plt.tight_layout()
    return fig

def plot_loss_curves(experiments):
    """Plot delle curve di loss durante training"""
    # Raggruppa per loss type
    loss_types = list(set([exp['loss'] for exp in experiments]))
    
    fig, axes = plt.subplots(1, len(loss_types), figsize=(6*len(loss_types), 5))
    if len(loss_types) == 1:
        axes = [axes]
    
    fig.suptitle('Curve di Training Loss per Loss Function', fontsize=16, fontweight='bold')
    
    for idx, loss_type in enumerate(loss_types):
        ax = axes[idx]
        
        for exp in experiments:
            if exp['loss'] == loss_type:
                history = exp['history']
                label = f"{exp['scenario']}"
                ax.plot(history['epoch'], history['train_loss'], 
                       label=f"{label} (train)", alpha=0.7, linewidth=2)
                ax.plot(history['epoch'], history['val_loss'], 
                       label=f"{label} (val)", alpha=0.7, linestyle='--', linewidth=2)
        
        ax.set_xlabel('Epoch', fontweight='bold')
        ax.set_ylabel('Loss', fontweight='bold')
        ax.set_title(f'Loss: {loss_type.upper()}', fontweight='bold')
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

# ============================================================================
# SEZIONE 3: ANALISI DISTRIBUZIONI
# ============================================================================

def plot_score_distributions(experiments):
    """Plot delle distribuzioni genuine vs impostor"""
    n_exp = len(experiments)
    n_cols = 3
    n_rows = (n_exp + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 5*n_rows))
    axes = axes.flatten() if n_exp > 1 else [axes]
    
    fig.suptitle('Distribuzioni Score: Genuine vs Impostor', fontsize=16, fontweight='bold')
    
    for idx, exp in enumerate(experiments):
        ax = axes[idx]
        m = exp['metrics']
        
        genuine = m['genuine_vals']
        impostor = m['impostor_vals']
        
        if len(genuine) > 0 and len(impostor) > 0:
            # Plot istogrammi
            ax.hist(genuine, bins=50, alpha=0.6, label='Genuine', 
                   color='green', density=True, edgecolor='black')
            ax.hist(impostor, bins=50, alpha=0.6, label='Impostor', 
                   color='red', density=True, edgecolor='black')
            
            # Aggiungi linee per media
            ax.axvline(m['mu_genuine'], color='darkgreen', 
                      linestyle='--', linewidth=2, label=f"μ_gen={m['mu_genuine']:.3f}")
            ax.axvline(m['mu_impostor'], color='darkred', 
                      linestyle='--', linewidth=2, label=f"μ_imp={m['mu_impostor']:.3f}")
            
            # Threshold EER
            ax.axvline(m['eer_threshold'], color='blue', 
                      linestyle=':', linewidth=2, label=f"EER thr={m['eer_threshold']:.3f}")
            
            ax.set_xlabel('Score', fontweight='bold')
            ax.set_ylabel('Density', fontweight='bold')
            ax.set_title(f"{exp['model'][:20]}...\n{exp['loss']} - {exp['scenario']}", 
                        fontweight='bold', fontsize=9)
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)
    
    # Nascondi assi extra
    for idx in range(n_exp, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    return fig

def plot_separation_quality(df_metrics):
    """Analisi qualità separazione distribuzioni"""
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot 1: D-prime per scenario
    ax = axes[0]
    pivot = df_metrics.pivot_table(values="D'", index='Scenario', columns='Loss')
    pivot.plot(kind='bar', ax=ax, width=0.8)
    ax.set_title("D-prime: Misura di Separabilità", fontweight='bold', fontsize=14)
    ax.set_xlabel('Scenario', fontweight='bold')
    ax.set_ylabel("D' value", fontweight='bold')
    ax.legend(title='Loss Function')
    ax.grid(True, alpha=0.3, axis='y')
    ax.axhline(y=1.0, color='red', linestyle='--', label="D'=1 (threshold)")
    
    # Plot 2: Decidability
    ax = axes[1]
    pivot = df_metrics.pivot_table(values='Decidability', index='Scenario', columns='Loss')
    pivot.plot(kind='bar', ax=ax, width=0.8)
    ax.set_title("Decidability Index", fontweight='bold', fontsize=14)
    ax.set_xlabel('Scenario', fontweight='bold')
    ax.set_ylabel('Decidability', fontweight='bold')
    ax.legend(title='Loss Function')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    return fig

# ============================================================================
# SEZIONE 4: CURVE ROC E DET
# ============================================================================

def plot_roc_curves(experiments):
    """Plot curve ROC per tutti gli esperimenti"""
    # Raggruppa per loss
    loss_types = list(set([exp['loss'] for exp in experiments]))
    
    fig, axes = plt.subplots(1, len(loss_types), figsize=(6*len(loss_types), 6))
    if len(loss_types) == 1:
        axes = [axes]
    
    fig.suptitle('Curve ROC per Loss Function', fontsize=16, fontweight='bold')
    
    for idx, loss_type in enumerate(loss_types):
        ax = axes[idx]
        
        for exp in experiments:
            if exp['loss'] == loss_type:
                m = exp['metrics']
                fpr = m['fpr']
                tpr = m['tpr']
                auc = m['auc']
                
                if len(fpr) > 0 and len(tpr) > 0:
                    ax.plot(fpr, tpr, linewidth=2, 
                           label=f"{exp['scenario']} (AUC={auc:.4f})")
        
        ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
        ax.set_xlabel('False Positive Rate', fontweight='bold')
        ax.set_ylabel('True Positive Rate', fontweight='bold')
        ax.set_title(f'ROC Curve - {loss_type.upper()}', fontweight='bold')
        ax.legend(loc='lower right', fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_xlim([0, 1])
        ax.set_ylim([0, 1])
    
    plt.tight_layout()
    return fig

def plot_det_curves(experiments):
    """Plot curve DET (Detection Error Tradeoff)"""
    loss_types = list(set([exp['loss'] for exp in experiments]))
    
    fig, axes = plt.subplots(1, len(loss_types), figsize=(6*len(loss_types), 6))
    if len(loss_types) == 1:
        axes = [axes]
    
    fig.suptitle('Curve DET (Detection Error Tradeoff)', fontsize=16, fontweight='bold')
    
    for idx, loss_type in enumerate(loss_types):
        ax = axes[idx]
        
        for exp in experiments:
            if exp['loss'] == loss_type:
                m = exp['metrics']
                fpr = m['fpr']
                tpr = m['tpr']
                
                if len(fpr) > 0 and len(tpr) > 0:
                    fnr = 1 - tpr  # False Negative Rate
                    ax.plot(fpr * 100, fnr * 100, linewidth=2,
                           label=f"{exp['scenario']}")
        
        ax.set_xlabel('False Acceptance Rate (%)', fontweight='bold')
        ax.set_ylabel('False Rejection Rate (%)', fontweight='bold')
        ax.set_title(f'DET Curve - {loss_type.upper()}', fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_xscale('log')
        ax.set_yscale('log')
    
    plt.tight_layout()
    return fig

# ============================================================================
# SEZIONE 5: ANALISI OPERATING POINTS
# ============================================================================

def plot_operating_points(df_metrics):
    """Analisi punti operativi standard"""
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # GAR at FAR = 0.1%
    ax = axes[0]
    pivot = df_metrics.pivot_table(values='GAR@FAR=0.1%', index='Scenario', columns='Loss')
    pivot.plot(kind='bar', ax=ax, width=0.8)
    ax.set_title('GAR @ FAR=0.1% (Security Applications)', fontweight='bold', fontsize=14)
    ax.set_xlabel('Scenario', fontweight='bold')
    ax.set_ylabel('GAR (%)', fontweight='bold')
    ax.legend(title='Loss Function')
    ax.grid(True, alpha=0.3, axis='y')
    ax.axhline(y=0.95, color='green', linestyle='--', label='Target 95%')
    
    # GAR at FAR = 1%
    ax = axes[1]
    pivot = df_metrics.pivot_table(values='GAR@FAR=1%', index='Scenario', columns='Loss')
    pivot.plot(kind='bar', ax=ax, width=0.8)
    ax.set_title('GAR @ FAR=1% (Convenience Applications)', fontweight='bold', fontsize=14)
    ax.set_xlabel('Scenario', fontweight='bold')
    ax.set_ylabel('GAR (%)', fontweight='bold')
    ax.legend(title='Loss Function')
    ax.grid(True, alpha=0.3, axis='y')
    ax.axhline(y=0.99, color='green', linestyle='--', label='Target 99%')
    
    plt.tight_layout()
    return fig

# ============================================================================
# SEZIONE 6: ANALISI CROSS-DATASET
# ============================================================================

def analyze_cross_dataset_performance(df_metrics):
    """Analizza performance cross-dataset vs same-dataset"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Crea colonna per tipo scenario
    df_metrics['scenario_type'] = df_metrics.apply(
        lambda x: 'Same Dataset' if x['Train'] == x['Test'] else 'Cross Dataset', 
        axis=1
    )
    
    metrics_to_plot = ['AUC', 'EER', 'F1', "D'"]
    
    for idx, metric in enumerate(metrics_to_plot):
        ax = axes[idx // 2, idx % 2]
        
        # Box plot per tipo scenario
        data_to_plot = []
        labels = []
        
        for loss in df_metrics['Loss'].unique():
            for stype in ['Same Dataset', 'Cross Dataset']:
                subset = df_metrics[(df_metrics['Loss'] == loss) & 
                                   (df_metrics['scenario_type'] == stype)]
                if len(subset) > 0:
                    data_to_plot.append(subset[metric].values)
                    labels.append(f"{loss}\n{stype}")
        
        bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True)
        
        # Colora box
        colors = ['lightblue', 'lightcoral'] * (len(data_to_plot) // 2 + 1)
        for patch, color in zip(bp['boxes'], colors[:len(bp['boxes'])]):
            patch.set_facecolor(color)
        
        ax.set_title(f'{metric}: Same vs Cross Dataset', fontweight='bold')
        ax.set_ylabel(metric, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')
        ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    return fig

# ============================================================================
# SEZIONE 7: STATISTICAL ANALYSIS
# ============================================================================

def perform_statistical_analysis(df_metrics):
    """Analisi statistica delle differenze tra loss functions"""
    print("\n" + "="*80)
    print("ANALISI STATISTICA COMPARATIVA")
    print("="*80)
    
    metrics = ['AUC', 'EER', 'F1', "D'"]
    loss_types = df_metrics['Loss'].unique()
    
    for metric in metrics:
        print(f"\n{metric}:")
        print("-" * 40)
        
        # Media e std per ogni loss
        for loss in loss_types:
            subset = df_metrics[df_metrics['Loss'] == loss][metric]
            print(f"  {loss:12s}: μ={subset.mean():.4f}, σ={subset.std():.4f}, "
                  f"min={subset.min():.4f}, max={subset.max():.4f}")
        
        # Test statistico (ANOVA se >2 gruppi)
        if len(loss_types) > 2:
            groups = [df_metrics[df_metrics['Loss'] == loss][metric].values 
                     for loss in loss_types]
            f_stat, p_value = stats.f_oneway(*groups)
            print(f"\n  ANOVA: F={f_stat:.4f}, p-value={p_value:.4f}")
            if p_value < 0.05:
                print("  → Differenze statisticamente significative!")
            else:
                print("  → Nessuna differenza statisticamente significativa")

def create_ranking_table(df_metrics):
    """Crea tabella di ranking per ogni metrica"""
    print("\n" + "="*80)
    print("RANKING CONFIGURAZIONI")
    print("="*80)
    
    metrics = ['AUC', 'EER', 'F1', "D'", 'GAR@FAR=0.1%']
    
    for metric in metrics:
        print(f"\n{metric}:")
        print("-" * 80)
        
        # Ordina (decrescente per tutto tranne EER)
        ascending = (metric == 'EER')
        ranked = df_metrics.sort_values(metric, ascending=ascending)
        
        for i, (_, row) in enumerate(ranked.head(10).iterrows(), 1):
            print(f"  {i:2d}. {row['Loss']:12s} | {row['Scenario']:20s} | "
                  f"{metric}={row[metric]:.4f}")

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    print("="*80)
    print("ANALISI COMPLETA RISULTATI ESPERIMENTI BIOMETRICI")
    print("="*80)
    
    # 1. Carica dati
    print("\n[1/8] Caricamento dati...")
    experiments = load_experiment_data('../results')
    print(f"   → Caricati {len(experiments)} esperimenti")
    
    # 2. Crea DataFrame metriche
    print("\n[2/8] Creazione DataFrame comparativo...")
    df_metrics = create_metrics_comparison(experiments)
    print(f"   → DataFrame con {len(df_metrics)} righe e {len(df_metrics.columns)} colonne")
    
    # 3. Plot comparazione metriche
    print("\n[3/8] Generazione plot comparazione metriche...")
    fig1 = plot_metrics_comparison(df_metrics)
    plt.savefig('01_metrics_comparison.png', dpi=300, bbox_inches='tight')
    print("   → Salvato: 01_metrics_comparison.png")
    
    # 4. Plot curve loss
    print("\n[4/8] Generazione curve di training...")
    fig2 = plot_loss_curves(experiments)
    plt.savefig('02_training_curves.png', dpi=300, bbox_inches='tight')
    print("   → Salvato: 02_training_curves.png")
    
    # 5. Plot distribuzioni
    print("\n[5/8] Generazione distribuzioni score...")
    fig3 = plot_score_distributions(experiments)
    plt.savefig('03_score_distributions.png', dpi=300, bbox_inches='tight')
    print("   → Salvato: 03_score_distributions.png")
    
    # 6. Plot separazione
    print("\n[6/8] Analisi qualità separazione...")
    fig4 = plot_separation_quality(df_metrics)
    plt.savefig('04_separation_quality.png', dpi=300, bbox_inches='tight')
    print("   → Salvato: 04_separation_quality.png")
    
    # 7. Curve ROC
    print("\n[7/8] Generazione curve ROC...")
    fig5 = plot_roc_curves(experiments)
    plt.savefig('05_roc_curves.png', dpi=300, bbox_inches='tight')
    print("   → Salvato: 05_roc_curves.png")
    
    # 8. Curve DET
    print("\n[8/8] Generazione curve DET...")
    fig6 = plot_det_curves(experiments)
    plt.savefig('06_det_curves.png', dpi=300, bbox_inches='tight')
    print("   → Salvato: 06_det_curves.png")
    
    # 9. Operating points
    print("\n[9/10] Analisi operating points...")
    fig7 = plot_operating_points(df_metrics)
    plt.savefig('07_operating_points.png', dpi=300, bbox_inches='tight')
    print("   → Salvato: 07_operating_points.png")
    
    # 10. Cross-dataset analysis
    print("\n[10/11] Analisi cross-dataset...")
    fig8 = analyze_cross_dataset_performance(df_metrics)
    plt.savefig('08_cross_dataset_analysis.png', dpi=300, bbox_inches='tight')
    print("   → Salvato: 08_cross_dataset_analysis.png")
    
    # 11. Analisi statistiche
    print("\n[11/11] Esecuzione analisi statistiche...")
    perform_statistical_analysis(df_metrics)
    create_ranking_table(df_metrics)
    
    # 12. Salva risultati
    print("\n[12/12] Salvataggio risultati...")
    df_metrics.to_csv('results_summary.csv', index=False)
    print("   → Salvato: results_summary.csv")
    
    print("\n" + "="*80)
    print("ANALISI COMPLETATA!")
    print("="*80)
    print(f"\nFile generati:")
    print("  - 01_metrics_comparison.png")
    print("  - 02_training_curves.png")
    print("  - 03_score_distributions.png")
    print("  - 04_separation_quality.png")
    print("  - 05_roc_curves.png")
    print("  - 06_det_curves.png")
    print("  - 07_operating_points.png")
    print("  - 08_cross_dataset_analysis.png")
    print("  - results_summary.csv")
    
    plt.show()
    
    return experiments, df_metrics

# Esegui analisi
if __name__ == "__main__":
    experiments, df_metrics = main()

ANALISI COMPLETA RISULTATI ESPERIMENTI BIOMETRICI

[1/8] Caricamento dati...


ValueError: could not convert string to float: '...'